# 🧪 Sesión 26 · Transacciones ACID, Aislamiento y Rollback

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Qué pasa si falla la segunda mitad de una operación?

---

## 🖥️ Así se verá el resultado final

```http
GET /api/libros/1                 -> 200 {"ejemplares":1}
POST /api/prestamos {"libro_id":1,"usuario":"ana"}
                                   -> 201 {"prestamo_id":1,"quedan":0}
GET /api/libros/1                 -> 200 {"ejemplares":0}
POST /api/prestamos {"libro_id":1,"usuario":"luis"}
                                   -> 409 {"error":"sin ejemplares"}
GET /api/libros/1                 -> 200 {"ejemplares":0}  # no bajó a -1
```

_Si el segundo préstamo descontara antes de validar, el stock quedaría en -1. La transacción lo impide._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Agrupar varias escrituras en una transacción.
- Hacer rollback cuando una condición de negocio falla.
- Verificar que el stock no queda inconsistente.

## ✅ Criterios de aceptación

El verificador ejecutará los casos HTTP y comprobará los archivos del contrato. Todo debe quedar en verde:

- [ ] El libro inicia con un ejemplar → HTTP 200
- [ ] El primer préstamo se completa → HTTP 201
- [ ] El stock quedó en cero → HTTP 200
- [ ] El segundo préstamo falla con 409 → HTTP 409
- [ ] El stock no quedó negativo → HTTP 200
- [ ] La operación usa una transacción

## 🧩 El reto

Implementa un préstamo de libros que descuente stock y registre el préstamo en una sola transacción.

**Requisitos del contrato**

- La base inicia con el libro 1, «Rayuela», con un ejemplar disponible.
- POST /api/prestamos descuenta un ejemplar y registra el préstamo.
- Si no hay ejemplares, responde 409 y no modifica el stock.
- GET /api/libros/{id} muestra los ejemplares restantes.
- No puede quedar stock negativo ni un préstamo sin descuento.

> **Pista conceptual:** Valida y escribe dentro de la misma transacción. Si haces `commit` antes de comprobar la condición, ya no puedes deshacer el descuento.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion26.md` con el caso de rollback.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [ ]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/main/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [ ]:
%%writefile app.py
from fastapi import FastAPI

app = FastAPI(title="Tu préstamo transaccional se verá así")

# Implementa aquí el contrato declarado en el reto.
# Tú decides los modelos, las rutas, las validaciones y el manejo de errores.


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [ ]:
from verificar import probar_http

casos = [{'descripcion': 'El libro inicia con un ejemplar',
  'metodo': 'GET',
  'ruta': '/api/libros/1',
  'estado': 200,
  'json_contiene': {'id': 1, 'ejemplares': 1}},
 {'descripcion': 'El primer préstamo se completa',
  'metodo': 'POST',
  'ruta': '/api/prestamos',
  'json': {'libro_id': 1, 'usuario': 'ana'},
  'estado': 201,
  'json_contiene': {'libro_id': 1, 'quedan': 0}},
 {'descripcion': 'El stock quedó en cero',
  'metodo': 'GET',
  'ruta': '/api/libros/1',
  'estado': 200,
  'json_contiene': {'ejemplares': 0}},
 {'descripcion': 'El segundo préstamo falla con 409',
  'metodo': 'POST',
  'ruta': '/api/prestamos',
  'json': {'libro_id': 1, 'usuario': 'luis'},
  'estado': 409},
 {'descripcion': 'El stock no quedó negativo',
  'metodo': 'GET',
  'ruta': '/api/libros/1',
  'estado': 200,
  'json_contiene': {'ejemplares': 0}}]

probar_http(casos, archivo="app.py", limpiar=['biblioteca.db'], env={})


In [ ]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("app.py", regex=["commit|rollback"], descripcion="La operación usa una transacción"))
resumen(resultados)


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion26.md` a `auditorias/auditoria_sesion26.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
